In [1]:
from __future__ import annotations

import gc
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

AMAZON_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "amazon"
    / "amazon_price_training.parquet"
)

FEATURE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "features"
)

TEXT_PCA_FILE = (
    FEATURE_ROOT
    / "amazon_text_pca.npy"
)

CLEAN_FEATURE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "features_clean"
)

CLEAN_MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "clustering_clean"
)

CLEAN_CLUSTER_ROOT = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "clean"
)

for folder in [
    CLEAN_FEATURE_ROOT,
    CLEAN_MODEL_ROOT,
    CLEAN_CLUSTER_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# OUTPUT FILES
# ============================================================

CLEAN_STRUCTURED_FILE = (
    CLEAN_FEATURE_ROOT
    / "amazon_clean_structured_features.npy"
)

CLEAN_CLUSTER_FEATURE_FILE = (
    CLEAN_FEATURE_ROOT
    / "amazon_clean_cluster_features.npy"
)

CLEAN_CLUSTERED_DATA_FILE = (
    CLEAN_CLUSTER_ROOT
    / "amazon_products_clustered_clean.parquet"
)

SCALER_FILE = (
    CLEAN_MODEL_ROOT
    / "structured_scaler_clean.joblib"
)

KMEANS_FILE = (
    CLEAN_MODEL_ROOT
    / "minibatch_kmeans_clean.joblib"
)

CLUSTER_SUMMARY_FILE = (
    CLEAN_CLUSTER_ROOT
    / "clean_cluster_summary.csv"
)


# ============================================================
# SETTINGS
# ============================================================

N_CLUSTERS = 100

BATCH_SIZE = 20_000

RANDOM_STATE = 42


# ============================================================
# LOAD AMAZON METADATA
# ============================================================

print("=" * 90)
print("LOADING AMAZON DATA")
print("=" * 90)

amazon_df = pd.read_parquet(
    AMAZON_FILE,
    columns=[
        "asin",
        "stars",
        "reviews",
        "boughtInLastMonth",
        "isBestSeller",
    ],
)

print(
    "Amazon rows:",
    f"{len(amazon_df):,}"
)


# ============================================================
# CREATE CLEAN STRUCTURED FEATURES
# ============================================================

print()
print("=" * 90)
print("CREATING CLEAN STRUCTURED FEATURES")
print("=" * 90)


stars = (
    pd.to_numeric(
        amazon_df["stars"],
        errors="coerce",
    )
    .fillna(0)
    .to_numpy(
        dtype=np.float32
    )
)


reviews = (
    pd.to_numeric(
        amazon_df["reviews"],
        errors="coerce",
    )
    .fillna(0)
    .to_numpy(
        dtype=np.float32
    )
)


bought = (
    pd.to_numeric(
        amazon_df[
            "boughtInLastMonth"
        ],
        errors="coerce",
    )
    .fillna(0)
    .to_numpy(
        dtype=np.float32
    )
)


is_best_seller = (
    amazon_df[
        "isBestSeller"
    ]
    .fillna(False)
    .astype(int)
    .to_numpy(
        dtype=np.float32
    )
)


reviews_log1p = np.log1p(
    reviews
).astype(np.float32)


bought_log1p = np.log1p(
    bought
).astype(np.float32)


clean_structured_raw = np.column_stack(
    [
        stars,
        reviews_log1p,
        bought_log1p,
        is_best_seller,
    ]
).astype(np.float32)


print(
    "Raw structured shape:",
    clean_structured_raw.shape
)


# ============================================================
# SCALE STRUCTURED FEATURES
# ============================================================

print()
print("=" * 90)
print("FITTING CLEAN STRUCTURED SCALER")
print("=" * 90)


scaler = StandardScaler()

clean_structured_scaled = (
    scaler
    .fit_transform(
        clean_structured_raw
    )
    .astype(np.float32)
)


np.save(
    CLEAN_STRUCTURED_FILE,
    clean_structured_scaled,
)


joblib.dump(
    scaler,
    SCALER_FILE,
)


print(
    "Scaled structured shape:",
    clean_structured_scaled.shape
)

print(
    "Scaler saved:",
    SCALER_FILE
)


# ============================================================
# LOAD TEXT PCA AS MEMORY MAP
# ============================================================

print()
print("=" * 90)
print("LOADING EXISTING TEXT PCA")
print("=" * 90)


text_pca = np.load(
    TEXT_PCA_FILE,
    mmap_mode="r",
)


print(
    "Text PCA shape:",
    text_pca.shape
)


if len(text_pca) != len(amazon_df):
    raise RuntimeError(
        "Text PCA and Amazon dataset "
        "row counts do not match."
    )


if text_pca.shape[1] != 64:
    raise RuntimeError(
        f"Expected 64 PCA features, "
        f"found {text_pca.shape[1]}"
    )


# ============================================================
# CREATE 68-D FEATURE FILE USING MEMORY MAP
# ============================================================

print()
print("=" * 90)
print("CREATING CLEAN 68-D CLUSTER FEATURES")
print("=" * 90)


n_rows = len(amazon_df)

FINAL_DIM = (
    text_pca.shape[1]
    + clean_structured_scaled.shape[1]
)

print(
    "Final feature dimension:",
    FINAL_DIM
)


if FINAL_DIM != 68:
    raise RuntimeError(
        f"Expected 68 features, "
        f"got {FINAL_DIM}"
    )


clean_cluster_features = (
    np.lib.format.open_memmap(
        CLEAN_CLUSTER_FEATURE_FILE,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_rows,
            FINAL_DIM,
        ),
    )
)


for start in tqdm(
    range(
        0,
        n_rows,
        BATCH_SIZE,
    ),
    desc="Building clean features",
):

    end = min(
        start + BATCH_SIZE,
        n_rows,
    )

    clean_cluster_features[
        start:end,
        :64,
    ] = text_pca[
        start:end
    ]

    clean_cluster_features[
        start:end,
        64:68,
    ] = clean_structured_scaled[
        start:end
    ]


clean_cluster_features.flush()


print(
    "Clean cluster feature shape:",
    clean_cluster_features.shape
)


# ============================================================
# TRAIN CLEAN MINIBATCH KMEANS
# ============================================================

print()
print("=" * 90)
print("TRAINING CLEAN MINIBATCH KMEANS")
print("=" * 90)


kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,

    batch_size=BATCH_SIZE,

    random_state=RANDOM_STATE,

    n_init="auto",

    reassignment_ratio=0.01,

    max_no_improvement=20,

    verbose=0,
)


# ------------------------------------------------------------
# PARTIAL FIT
# ------------------------------------------------------------

for start in tqdm(
    range(
        0,
        n_rows,
        BATCH_SIZE,
    ),
    desc="Training KMeans",
):

    end = min(
        start + BATCH_SIZE,
        n_rows,
    )

    batch = np.asarray(
        clean_cluster_features[
            start:end
        ],
        dtype=np.float32,
    )

    kmeans.partial_fit(
        batch
    )


joblib.dump(
    kmeans,
    KMEANS_FILE,
)


print(
    "KMeans saved:",
    KMEANS_FILE
)

print(
    "KMeans input features:",
    kmeans.n_features_in_
)

print(
    "Cluster centers:",
    kmeans.cluster_centers_.shape
)


# ============================================================
# PREDICT CLEAN CLUSTER IDS
# ============================================================

print()
print("=" * 90)
print("ASSIGNING CLEAN CLUSTERS")
print("=" * 90)


clean_cluster_ids = np.empty(
    n_rows,
    dtype=np.int16,
)


for start in tqdm(
    range(
        0,
        n_rows,
        BATCH_SIZE,
    ),
    desc="Predicting clusters",
):

    end = min(
        start + BATCH_SIZE,
        n_rows,
    )

    batch = np.asarray(
        clean_cluster_features[
            start:end
        ],
        dtype=np.float32,
    )

    clean_cluster_ids[
        start:end
    ] = kmeans.predict(
        batch
    ).astype(np.int16)


# ============================================================
# SAVE CLEAN CLUSTERED DATASET
# ============================================================

print()
print("=" * 90)
print("SAVING CLEAN CLUSTER IDS")
print("=" * 90)


clustered_df = pd.DataFrame(
    {
        "asin":
            amazon_df["asin"].values,

        "clean_cluster_id":
            clean_cluster_ids,
    }
)


clustered_df.to_parquet(
    CLEAN_CLUSTERED_DATA_FILE,
    index=False,
    compression="snappy",
)


print(
    "Saved:",
    CLEAN_CLUSTERED_DATA_FILE
)


# ============================================================
# VALIDATION
# ============================================================

print()
print("=" * 90)
print("CLEAN CLUSTER VALIDATION")
print("=" * 90)


print(
    "Rows:",
    f"{len(clustered_df):,}"
)

print(
    "Unique ASIN:",
    f"{clustered_df['asin'].nunique():,}"
)

print(
    "Clusters:",
    clustered_df[
        "clean_cluster_id"
    ].nunique()
)

print(
    "Missing cluster IDs:",
    clustered_df[
        "clean_cluster_id"
    ].isna().sum()
)


cluster_counts = (
    clustered_df[
        "clean_cluster_id"
    ]
    .value_counts()
    .sort_index()
)


print()
print(
    "Products per cluster:"
)

display(
    cluster_counts.describe()
)


# ============================================================
# CLUSTER SUMMARY
# ============================================================

cluster_summary = (
    clustered_df
    .groupby(
        "clean_cluster_id"
    )
    .size()
    .reset_index(
        name="product_count"
    )
)


cluster_summary.to_csv(
    CLUSTER_SUMMARY_FILE,
    index=False,
)


# ============================================================
# SAFETY CHECK
# ============================================================

assert (
    len(clustered_df)
    == len(amazon_df)
)

assert (
    clustered_df["asin"]
    .duplicated()
    .sum()
    == 0
)

assert (
    clustered_df[
        "clean_cluster_id"
    ]
    .nunique()
    == N_CLUSTERS
)

assert (
    kmeans.n_features_in_
    == 68
)


print()
print("=" * 90)
print("CLEAN CLUSTERING COMPLETED")
print("=" * 90)

print(
    "✅ NO price used"
)

print(
    "✅ NO price_band used"
)

print(
    "✅ NO listPrice used"
)

print(
    "✅ KMeans input dimension = 68"
)

print(
    "✅ Clean clusters =",
    N_CLUSTERS
)

print()
print(
    "Scaler:",
    SCALER_FILE
)

print(
    "KMeans:",
    KMEANS_FILE
)

print(
    "Cluster mapping:",
    CLEAN_CLUSTERED_DATA_FILE
)

LOADING AMAZON DATA
Amazon rows: 1,393,564

CREATING CLEAN STRUCTURED FEATURES
Raw structured shape: (1393564, 4)

FITTING CLEAN STRUCTURED SCALER
Scaled structured shape: (1393564, 4)
Scaler saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/clustering_clean/structured_scaler_clean.joblib

LOADING EXISTING TEXT PCA
Text PCA shape: (1393564, 64)

CREATING CLEAN 68-D CLUSTER FEATURES
Final feature dimension: 68


Building clean features:   0%|          | 0/70 [00:00<?, ?it/s]

Clean cluster feature shape: (1393564, 68)

TRAINING CLEAN MINIBATCH KMEANS


Training KMeans:   0%|          | 0/70 [00:00<?, ?it/s]

KMeans saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/clustering_clean/minibatch_kmeans_clean.joblib
KMeans input features: 68
Cluster centers: (100, 68)

ASSIGNING CLEAN CLUSTERS


Predicting clusters:   0%|          | 0/70 [00:00<?, ?it/s]


SAVING CLEAN CLUSTER IDS
Saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/clustering/amazon/clean/amazon_products_clustered_clean.parquet

CLEAN CLUSTER VALIDATION
Rows: 1,393,564
Unique ASIN: 1,393,564
Clusters: 100
Missing cluster IDs: 0

Products per cluster:


count      100.000000
mean     13935.640000
std      11761.857797
min        172.000000
25%       5551.500000
50%      10326.500000
75%      17499.750000
max      52169.000000
Name: count, dtype: float64


CLEAN CLUSTERING COMPLETED
✅ NO price used
✅ NO price_band used
✅ NO listPrice used
✅ KMeans input dimension = 68
✅ Clean clusters = 100

Scaler: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/clustering_clean/structured_scaler_clean.joblib
KMeans: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/clustering_clean/minibatch_kmeans_clean.joblib
Cluster mapping: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/clustering/amazon/clean/amazon_products_clustered_clean.parquet


## Step 1 — Attach clean cluster IDs to your splits

In [2]:
from pathlib import Path
import pandas as pd


# ============================================================
# PATHS
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

CLEAN_CLUSTER_FILE = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "clean"
    / "amazon_products_clustered_clean.parquet"
)

CLEAN_MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input_clean_cluster"
)

CLEAN_MODEL_INPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD CLEAN CLUSTERS
# ============================================================

clean_clusters = pd.read_parquet(
    CLEAN_CLUSTER_FILE
)

print("=" * 80)
print("CLEAN CLUSTER MAPPING")
print("=" * 80)

print(
    "Rows:",
    f"{len(clean_clusters):,}"
)

print(
    "Unique ASINs:",
    f"{clean_clusters['asin'].nunique():,}"
)

print(
    "Clusters:",
    clean_clusters[
        "clean_cluster_id"
    ].nunique()
)


# ============================================================
# REPLACE OLD CLUSTER ID
# ============================================================

for split_name in [
    "train",
    "validation",
    "test",
]:

    print()
    print("=" * 80)
    print(
        f"PROCESSING {split_name.upper()}"
    )
    print("=" * 80)

    input_file = (
        MODEL_INPUT_ROOT
        / f"{split_name}.parquet"
    )

    df = pd.read_parquet(
        input_file
    )

    original_rows = len(df)

    # Keep old cluster only for comparison/debugging
    if "cluster_id" in df.columns:
        df = df.rename(
            columns={
                "cluster_id":
                    "old_leaked_cluster_id"
            }
        )

    df = df.merge(
        clean_clusters,
        on="asin",
        how="left",
        validate="one_to_one",
    )

    missing_clusters = (
        df["clean_cluster_id"]
        .isna()
        .sum()
    )

    if missing_clusters > 0:
        raise RuntimeError(
            f"{split_name}: "
            f"{missing_clusters} products "
            f"missing clean cluster IDs."
        )

    df["cluster_id"] = (
        df["clean_cluster_id"]
        .astype(int)
    )

    output_file = (
        CLEAN_MODEL_INPUT_ROOT
        / f"{split_name}.parquet"
    )

    df.to_parquet(
        output_file,
        index=False,
        compression="snappy",
    )

    print(
        "Rows:",
        f"{len(df):,}"
    )

    print(
        "Original rows:",
        f"{original_rows:,}"
    )

    print(
        "Missing clean clusters:",
        missing_clusters
    )

    print(
        "Unique clean clusters:",
        df["cluster_id"].nunique()
    )

    print(
        "Saved:",
        output_file
    )


print()
print("=" * 80)
print("CLEAN MODEL SPLITS CREATED")
print("=" * 80)

CLEAN CLUSTER MAPPING
Rows: 1,393,564
Unique ASINs: 1,393,564
Clusters: 100

PROCESSING TRAIN
Rows: 13,984
Original rows: 13,984
Missing clean clusters: 0
Unique clean clusters: 100
Saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input_clean_cluster/train.parquet

PROCESSING VALIDATION
Rows: 2,997
Original rows: 2,997
Missing clean clusters: 0
Unique clean clusters: 98
Saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input_clean_cluster/validation.parquet

PROCESSING TEST
Rows: 2,997
Original rows: 2,997
Missing clean clusters: 0
Unique clean clusters: 99
Saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input_clean_cluster/test.parquet

CLEAN MODEL SPLITS CREATED


## Step 2 — Retrain the real final model

In [3]:
from __future__ import annotations

import re
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# PATHS
# ============================================================

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input_clean_cluster"
)

FINAL_MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "final_clean_v1"
)

FINAL_REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "final_clean_v1"
)

FINAL_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD
# ============================================================

train_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "train.parquet"
)

validation_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "validation.parquet"
)

test_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "test.parquet"
)


print("=" * 80)
print("FINAL CLEAN DATA")
print("=" * 80)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def extract_first_number(text):

    numbers = re.findall(
        r"\d+(?:\.\d+)?",
        str(text),
    )

    if not numbers:
        return 0.0

    try:
        return float(numbers[0])
    except Exception:
        return 0.0


def create_features(df):

    df = df.copy()

    df["title"] = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )

    df["stars"] = (
        pd.to_numeric(
            df["stars"],
            errors="coerce",
        )
        .fillna(0)
    )

    reviews = (
        pd.to_numeric(
            df["reviews"],
            errors="coerce",
        )
        .fillna(0)
    )

    bought = (
        pd.to_numeric(
            df["boughtInLastMonth"],
            errors="coerce",
        )
        .fillna(0)
    )

    df["reviews_log1p"] = (
        np.log1p(reviews)
    )

    df["bought_log1p"] = (
        np.log1p(bought)
    )

    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )

    df["cluster_id"] = (
        pd.to_numeric(
            df["cluster_id"],
            errors="coerce",
        )
        .fillna(-1)
        .astype(int)
    )

    # --------------------------------------------------------
    # TITLE STATS
    # --------------------------------------------------------

    df["title_char_length"] = (
        df["title"].str.len()
    )

    df["title_word_count"] = (
        df["title"]
        .str.split()
        .str.len()
        .fillna(0)
    )

    df["title_digit_count"] = (
        df["title"]
        .str.count(r"\d")
    )

    df["title_uppercase_count"] = (
        df["title"]
        .apply(
            lambda text:
            sum(
                char.isupper()
                for char in text
            )
        )
    )

    df["title_first_number"] = (
        df["title"]
        .apply(
            extract_first_number
        )
    )

    lower_title = (
        df["title"]
        .str.lower()
    )

    patterns = {

        "has_gb":
            r"\b\d+(?:\.\d+)?\s*gb\b",

        "has_tb":
            r"\b\d+(?:\.\d+)?\s*tb\b",

        "has_ram":
            r"\b(?:ram|memory)\b",

        "has_inch":
            r'\b\d+(?:\.\d+)?\s*(?:inch|inches|")',

        "has_cm":
            r"\b\d+(?:\.\d+)?\s*cm\b",

        "has_kg":
            r"\b\d+(?:\.\d+)?\s*kg\b",

        "has_gram":
            r"\b\d+(?:\.\d+)?\s*(?:g|gram|grams)\b",

        "has_watt":
            r"\b\d+(?:\.\d+)?\s*(?:w|watt|watts)\b",

        "has_volt":
            r"\b\d+(?:\.\d+)?\s*(?:v|volt|volts)\b",

        "has_pack":
            r"\b(?:pack|set|pair|bundle)\b",

        "has_multipack_number":
            r"\b\d+\s*[- ]?(?:pack|piece|pcs|count|ct)\b",

        "has_pro":
            r"\bpro\b",

        "has_max":
            r"\bmax\b",

        "has_premium":
            r"\bpremium\b",

        "has_professional":
            r"\bprofessional\b",

        "has_wireless":
            r"\bwireless\b",

        "has_smart":
            r"\bsmart\b",
    }

    for name, pattern in patterns.items():

        df[name] = (
            lower_title
            .str.contains(
                pattern,
                regex=True,
            )
            .astype(int)
        )

    return df


train_df = create_features(train_df)
validation_df = create_features(validation_df)
test_df = create_features(test_df)


# ============================================================
# FEATURE CONTRACT
# ============================================================

NUMERIC_FEATURES = [

    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",

    "title_char_length",
    "title_word_count",
    "title_digit_count",
    "title_uppercase_count",
    "title_first_number",

    "has_gb",
    "has_tb",
    "has_ram",

    "has_inch",
    "has_cm",

    "has_kg",
    "has_gram",

    "has_watt",
    "has_volt",

    "has_pack",
    "has_multipack_number",

    "has_pro",
    "has_max",
    "has_premium",
    "has_professional",
    "has_wireless",
    "has_smart",

    "cluster_id",
]

CATEGORICAL_FEATURES = [
    "category_name",
]


# ============================================================
# LEAKAGE CHECK
# ============================================================

FORBIDDEN = {
    "price",
    "price_band",
    "price_log1p",
    "listPrice",
    "discount_amount",
    "discount_percentage",
    "old_leaked_cluster_id",
}

used = set(
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

bad = used & FORBIDDEN

if bad:
    raise RuntimeError(
        f"Leakage found: {bad}"
    )

print(
    "✅ Feature contract contains "
    "no direct leakage."
)


# ============================================================
# PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)


X_train_structured = (
    preprocessor
    .fit_transform(train_df)
)

X_validation_structured = (
    preprocessor
    .transform(validation_df)
)

X_test_structured = (
    preprocessor
    .transform(test_df)
)


# ============================================================
# TF-IDF
# ============================================================

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",

    ngram_range=(1, 2),

    min_df=3,
    max_df=0.98,

    max_features=8000,

    sublinear_tf=True,

    dtype=np.float32,
)


X_train_tfidf = (
    tfidf.fit_transform(
        train_df["title"]
    )
)

X_validation_tfidf = (
    tfidf.transform(
        validation_df["title"]
    )
)

X_test_tfidf = (
    tfidf.transform(
        test_df["title"]
    )
)


# ============================================================
# COMBINE
# ============================================================

X_train = hstack(
    [
        csr_matrix(
            X_train_structured
        ),
        X_train_tfidf,
    ],
    format="csr",
)

X_validation = hstack(
    [
        csr_matrix(
            X_validation_structured
        ),
        X_validation_tfidf,
    ],
    format="csr",
)

X_test = hstack(
    [
        csr_matrix(
            X_test_structured
        ),
        X_test_tfidf,
    ],
    format="csr",
)


print()
print("Structured:", X_train_structured.shape)
print("TF-IDF:", X_train_tfidf.shape)
print("Final:", X_train.shape)


# ============================================================
# TARGET
# ============================================================

y_train = (
    train_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_validation = (
    validation_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_test = (
    test_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_train_log = (
    np.log1p(y_train)
)

y_validation_log = (
    np.log1p(y_validation)
)


# ============================================================
# TRAIN
# ============================================================

print()
print("=" * 80)
print(
    "TRAINING FINAL TRULY CLEAN MODEL"
)
print("=" * 80)


model = lgb.LGBMRegressor(

    objective="regression",

    n_estimators=4000,

    learning_rate=0.025,

    num_leaves=63,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,

    colsample_bytree=0.85,

    reg_alpha=0.05,

    reg_lambda=1.0,

    random_state=42,

    n_jobs=-1,

    verbosity=-1,
)


model.fit(

    X_train,

    y_train_log,

    eval_set=[
        (
            X_validation,
            y_validation_log,
        )
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=150,
            verbose=False,
        )
    ],
)


# ============================================================
# PREDICT
# ============================================================

validation_prediction = (
    np.expm1(
        model.predict(
            X_validation
        )
    )
)

test_prediction = (
    np.expm1(
        model.predict(
            X_test
        )
    )
)

validation_prediction = np.clip(
    validation_prediction,
    0,
    None,
)

test_prediction = np.clip(
    test_prediction,
    0,
    None,
)


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    actual,
    predicted,
):

    return {

        "mae":
            mean_absolute_error(
                actual,
                predicted,
            ),

        "rmse":
            np.sqrt(
                mean_squared_error(
                    actual,
                    predicted,
                )
            ),

        "median_ae":
            median_absolute_error(
                actual,
                predicted,
            ),

        "r2":
            r2_score(
                actual,
                predicted,
            ),
    }


validation_metrics = (
    calculate_metrics(
        y_validation,
        validation_prediction,
    )
)

test_metrics = (
    calculate_metrics(
        y_test,
        test_prediction,
    )
)


print()
print("VALIDATION")
print("-" * 60)

for name, value in (
    validation_metrics.items()
):
    print(
        f"{name:12s}: {value:.4f}"
    )


print()
print("TEST")
print("-" * 60)

for name, value in (
    test_metrics.items()
):
    print(
        f"{name:12s}: {value:.4f}"
    )


# ============================================================
# SAVE FINAL ARTIFACTS
# ============================================================

joblib.dump(
    model,
    FINAL_MODEL_ROOT
    / "price_model.joblib",
)

joblib.dump(
    preprocessor,
    FINAL_MODEL_ROOT
    / "structured_preprocessor.joblib",
)

joblib.dump(
    tfidf,
    FINAL_MODEL_ROOT
    / "title_tfidf.joblib",
)


prediction_df = pd.DataFrame(
    {
        "asin":
            test_df["asin"].values,

        "actual_price":
            y_test,

        "predicted_price":
            test_prediction,

        "absolute_error":
            np.abs(
                y_test
                - test_prediction
            ),

        "clean_cluster_id":
            test_df["cluster_id"].values,

        "price_band":
            test_df[
                "price_band"
            ].values,
    }
)


prediction_df.to_csv(
    FINAL_REPORT_ROOT
    / "test_predictions.csv",
    index=False,
)


results_df = pd.DataFrame(
    [
        {
            "dataset":
                "validation",

            **validation_metrics,
        },

        {
            "dataset":
                "test",

            **test_metrics,
        },
    ]
)


results_df.to_csv(
    FINAL_REPORT_ROOT
    / "final_metrics.csv",
    index=False,
)


print()
print("=" * 80)
print("FINAL CLEAN MODEL COMPLETED")
print("=" * 80)

print(
    "Model:",
    FINAL_MODEL_ROOT
    / "price_model.joblib"
)

print(
    "Reports:",
    FINAL_REPORT_ROOT
)

FINAL CLEAN DATA
Train: (13984, 28)
Validation: (2997, 28)
Test: (2997, 28)
✅ Feature contract contains no direct leakage.

Structured: (13984, 267)
TF-IDF: (13984, 8000)
Final: (13984, 8267)

TRAINING FINAL TRULY CLEAN MODEL


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



VALIDATION
------------------------------------------------------------
mae         : 27.2377
rmse        : 99.7261
median_ae   : 9.2715
r2          : 0.2004

TEST
------------------------------------------------------------
mae         : 26.1557
rmse        : 75.2317
median_ae   : 9.3024
r2          : 0.3700

FINAL CLEAN MODEL COMPLETED
Model: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/final_clean_v1/price_model.joblib
Reports: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/reports/price_prediction/final_clean_v1


## 1. Create the production inference module

In [9]:
from __future__ import annotations

import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

from scipy.sparse import csr_matrix, hstack
from sentence_transformers import SentenceTransformer


# ============================================================
# PROJECT PATH - JUPYTER SAFE
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

print("=" * 80)
print("PROJECT CONFIGURATION")
print("=" * 80)

print("Current directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)


# ============================================================
# PRICE MODEL FILES
# ============================================================

PRICE_MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "final_clean_v1"
)

PRICE_MODEL_FILE = (
    PRICE_MODEL_ROOT
    / "price_model.joblib"
)

PRICE_PREPROCESSOR_FILE = (
    PRICE_MODEL_ROOT
    / "structured_preprocessor.joblib"
)

PRICE_TFIDF_FILE = (
    PRICE_MODEL_ROOT
    / "title_tfidf.joblib"
)


# ============================================================
# CLEAN CLUSTER FILES
# ============================================================

CLUSTER_MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "clustering_clean"
)

CLUSTER_SCALER_FILE = (
    CLUSTER_MODEL_ROOT
    / "structured_scaler_clean.joblib"
)

CLUSTER_KMEANS_FILE = (
    CLUSTER_MODEL_ROOT
    / "minibatch_kmeans_clean.joblib"
)


# ============================================================
# TEXT PCA
# ============================================================

PCA_FILE = (
    PROJECT_ROOT
    / "models"
    / "clustering"
    / "incremental_pca.joblib"
)


# ============================================================
# CHECK REQUIRED FILES
# ============================================================

required_files = {
    "Price model":
        PRICE_MODEL_FILE,

    "Price preprocessor":
        PRICE_PREPROCESSOR_FILE,

    "TF-IDF":
        PRICE_TFIDF_FILE,

    "Clean cluster scaler":
        CLUSTER_SCALER_FILE,

    "Clean KMeans":
        CLUSTER_KMEANS_FILE,

    "Text PCA":
        PCA_FILE,
}


print()
print("=" * 80)
print("MODEL ARTIFACT CHECK")
print("=" * 80)

missing_files = []

for name, path in required_files.items():

    exists = path.exists()

    status = (
        "✅"
        if exists
        else "❌"
    )

    print(
        f"{status} {name}: {path}"
    )

    if not exists:
        missing_files.append(
            str(path)
        )


if missing_files:

    raise FileNotFoundError(
        "Missing required model files:\n"
        + "\n".join(
            missing_files
        )
    )


# ============================================================
# TEXT MODEL
# ============================================================

TEXT_MODEL_NAME = (
    "sentence-transformers/"
    "all-MiniLM-L6-v2"
)


# ============================================================
# DEVICE
# ============================================================

if torch.cuda.is_available():

    DEVICE = "cuda"

elif (
    hasattr(
        torch.backends,
        "mps"
    )
    and torch.backends.mps.is_available()
):

    DEVICE = "mps"

else:

    DEVICE = "cpu"


print()
print(
    "Using device:",
    DEVICE
)


# ============================================================
# LOAD MODELS
# ============================================================

print()
print("=" * 80)
print("LOADING PRODUCTION MODELS")
print("=" * 80)


print(
    "Loading price model..."
)

price_model = joblib.load(
    PRICE_MODEL_FILE
)


print(
    "Loading price preprocessor..."
)

price_preprocessor = joblib.load(
    PRICE_PREPROCESSOR_FILE
)


print(
    "Loading TF-IDF..."
)

price_tfidf = joblib.load(
    PRICE_TFIDF_FILE
)


print(
    "Loading clean cluster scaler..."
)

cluster_scaler = joblib.load(
    CLUSTER_SCALER_FILE
)


print(
    "Loading clean KMeans..."
)

cluster_kmeans = joblib.load(
    CLUSTER_KMEANS_FILE
)


print(
    "Loading text PCA..."
)

text_pca = joblib.load(
    PCA_FILE
)


print(
    "Loading SentenceTransformer..."
)

text_model = SentenceTransformer(
    TEXT_MODEL_NAME,
    device=DEVICE,
)


print()
print(
    "✅ Production inference models loaded"
)


# ============================================================
# MODEL VALIDATION
# ============================================================

print()
print("=" * 80)
print("MODEL DIMENSION CHECK")
print("=" * 80)


print(
    "PCA input:",
    getattr(
        text_pca,
        "n_features_in_",
        None,
    )
)

print(
    "PCA output:",
    getattr(
        text_pca,
        "n_components_",
        None,
    )
)

print(
    "Cluster scaler input:",
    getattr(
        cluster_scaler,
        "n_features_in_",
        None,
    )
)

print(
    "KMeans input:",
    getattr(
        cluster_kmeans,
        "n_features_in_",
        None,
    )
)

print(
    "Clusters:",
    getattr(
        cluster_kmeans,
        "n_clusters",
        None,
    )
)


if getattr(
    text_pca,
    "n_features_in_",
    None,
) != 384:

    raise RuntimeError(
        "PCA must expect 384 "
        "SentenceTransformer features."
    )


if getattr(
    text_pca,
    "n_components_",
    None,
) != 64:

    raise RuntimeError(
        "PCA must output 64 features."
    )


if getattr(
    cluster_scaler,
    "n_features_in_",
    None,
) != 4:

    raise RuntimeError(
        "Clean scaler must expect "
        "4 structured features."
    )


if getattr(
    cluster_kmeans,
    "n_features_in_",
    None,
) != 68:

    raise RuntimeError(
        "Clean KMeans must expect "
        "68 features."
    )


print(
    "✅ Model dimensions validated"
)


# ============================================================
# HELPER
# ============================================================

def extract_first_number(
    text: str,
) -> float:

    numbers = re.findall(
        r"\d+(?:\.\d+)?",
        str(text),
    )

    if not numbers:
        return 0.0

    try:
        return float(
            numbers[0]
        )

    except Exception:
        return 0.0


# ============================================================
# PRICE FEATURE ENGINEERING
# ============================================================

def create_price_features(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:

    df = dataframe.copy()


    # --------------------------------------------------------
    # BASIC TEXT
    # --------------------------------------------------------

    df["title"] = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )


    # --------------------------------------------------------
    # NUMERIC FEATURES
    # --------------------------------------------------------

    df["stars"] = (
        pd.to_numeric(
            df["stars"],
            errors="coerce",
        )
        .fillna(0)
    )


    reviews = (
        pd.to_numeric(
            df["reviews"],
            errors="coerce",
        )
        .fillna(0)
    )


    bought = (
        pd.to_numeric(
            df["boughtInLastMonth"],
            errors="coerce",
        )
        .fillna(0)
    )


    df["reviews_log1p"] = (
        np.log1p(
            reviews
        )
    )


    df["bought_log1p"] = (
        np.log1p(
            bought
        )
    )


    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )


    df["cluster_id"] = (
        pd.to_numeric(
            df["cluster_id"],
            errors="coerce",
        )
        .fillna(-1)
        .astype(int)
    )


    # --------------------------------------------------------
    # TITLE STATISTICS
    # --------------------------------------------------------

    df["title_char_length"] = (
        df["title"]
        .str.len()
    )


    df["title_word_count"] = (
        df["title"]
        .str.split()
        .str.len()
        .fillna(0)
    )


    df["title_digit_count"] = (
        df["title"]
        .str.count(
            r"\d"
        )
    )


    df["title_uppercase_count"] = (
        df["title"]
        .apply(
            lambda text:
            sum(
                char.isupper()
                for char in text
            )
        )
    )


    df["title_first_number"] = (
        df["title"]
        .apply(
            extract_first_number
        )
    )


    # --------------------------------------------------------
    # TITLE PATTERN FEATURES
    # --------------------------------------------------------

    lower_title = (
        df["title"]
        .str.lower()
    )


    patterns = {

        "has_gb":
            r"\b\d+(?:\.\d+)?\s*gb\b",

        "has_tb":
            r"\b\d+(?:\.\d+)?\s*tb\b",

        "has_ram":
            r"\b(?:ram|memory)\b",

        "has_inch":
            r'\b\d+(?:\.\d+)?\s*(?:inch|inches|")',

        "has_cm":
            r"\b\d+(?:\.\d+)?\s*cm\b",

        "has_kg":
            r"\b\d+(?:\.\d+)?\s*kg\b",

        "has_gram":
            r"\b\d+(?:\.\d+)?\s*(?:g|gram|grams)\b",

        "has_watt":
            r"\b\d+(?:\.\d+)?\s*(?:w|watt|watts)\b",

        "has_volt":
            r"\b\d+(?:\.\d+)?\s*(?:v|volt|volts)\b",

        "has_pack":
            r"\b(?:pack|set|pair|bundle)\b",

        "has_multipack_number":
            r"\b\d+\s*[- ]?"
            r"(?:pack|piece|pcs|count|ct)\b",

        "has_pro":
            r"\bpro\b",

        "has_max":
            r"\bmax\b",

        "has_premium":
            r"\bpremium\b",

        "has_professional":
            r"\bprofessional\b",

        "has_wireless":
            r"\bwireless\b",

        "has_smart":
            r"\bsmart\b",
    }


    for (
        feature_name,
        pattern,
    ) in patterns.items():

        df[
            feature_name
        ] = (
            lower_title
            .str.contains(
                pattern,
                regex=True,
            )
            .astype(int)
        )


    return df


# ============================================================
# CLEAN CLUSTER GENERATION
# ============================================================

def predict_clean_cluster(
    *,
    title: str,
    stars: float = 0,
    reviews: int = 0,
    bought_in_last_month: int = 0,
    is_best_seller: bool = False,
) -> int:

    if not str(
        title
    ).strip():

        raise ValueError(
            "Product title is required "
            "for cluster prediction."
        )


    # --------------------------------------------------------
    # SENTENCE TRANSFORMER
    # 1 x 384
    # --------------------------------------------------------

    embedding = (
        text_model.encode(
            [
                str(
                    title
                )
            ],

            convert_to_numpy=True,

            normalize_embeddings=False,

            show_progress_bar=False,
        )
        .astype(
            np.float32
        )
    )


    if embedding.shape != (
        1,
        384,
    ):

        raise RuntimeError(
            "Unexpected sentence "
            f"embedding shape: "
            f"{embedding.shape}"
        )


    # --------------------------------------------------------
    # PCA
    # 384 -> 64
    # --------------------------------------------------------

    text_features = (
        text_pca
        .transform(
            embedding
        )
        .astype(
            np.float32
        )
    )


    if text_features.shape != (
        1,
        64,
    ):

        raise RuntimeError(
            "Unexpected PCA shape: "
            f"{text_features.shape}"
        )


    # --------------------------------------------------------
    # CLEAN STRUCTURED FEATURES
    # --------------------------------------------------------

    stars_value = float(
        stars or 0
    )


    reviews_value = max(
        0.0,
        float(
            reviews or 0
        ),
    )


    bought_value = max(
        0.0,
        float(
            bought_in_last_month
            or 0
        ),
    )


    bestseller_value = float(
        bool(
            is_best_seller
        )
    )


    structured_raw = np.array(
        [
            [
                stars_value,

                np.log1p(
                    reviews_value
                ),

                np.log1p(
                    bought_value
                ),

                bestseller_value,
            ]
        ],

        dtype=np.float32,
    )


    # --------------------------------------------------------
    # SCALE 4 FEATURES
    # --------------------------------------------------------

    structured_scaled = (
        cluster_scaler
        .transform(
            structured_raw
        )
        .astype(
            np.float32
        )
    )


    if structured_scaled.shape != (
        1,
        4,
    ):

        raise RuntimeError(
            "Unexpected structured "
            f"cluster shape: "
            f"{structured_scaled.shape}"
        )


    # --------------------------------------------------------
    # 64 + 4 = 68
    # --------------------------------------------------------

    cluster_features = (
        np.concatenate(
            [
                text_features,
                structured_scaled,
            ],

            axis=1,
        )
        .astype(
            np.float32
        )
    )


    if cluster_features.shape != (
        1,
        68,
    ):

        raise RuntimeError(
            "Expected cluster features "
            f"(1, 68), got "
            f"{cluster_features.shape}"
        )


    # --------------------------------------------------------
    # KMEANS PREDICTION
    # --------------------------------------------------------

    cluster_id = int(
        cluster_kmeans
        .predict(
            cluster_features
        )[0]
    )


    return cluster_id


# ============================================================
# FINAL PRODUCT PRICE PREDICTION
# ============================================================

def predict_product_price(
    *,
    title: str,
    category_name: str,
    stars: float = 0,
    reviews: int = 0,
    bought_in_last_month: int = 0,
    is_best_seller: bool = False,
) -> dict:

    # --------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------

    title = str(
        title
    ).strip()

    category_name = str(
        category_name
    ).strip()


    if not title:

        raise ValueError(
            "Product title is required."
        )


    if not category_name:

        raise ValueError(
            "Category is required."
        )


    if stars < 0 or stars > 5:

        raise ValueError(
            "Stars must be between "
            "0 and 5."
        )


    if reviews < 0:

        raise ValueError(
            "Reviews cannot be negative."
        )


    if bought_in_last_month < 0:

        raise ValueError(
            "Bought in last month "
            "cannot be negative."
        )


    # --------------------------------------------------------
    # AUTOMATIC CLEAN CLUSTER
    # --------------------------------------------------------

    cluster_id = (
        predict_clean_cluster(
            title=title,

            stars=stars,

            reviews=reviews,

            bought_in_last_month=(
                bought_in_last_month
            ),

            is_best_seller=(
                is_best_seller
            ),
        )
    )


    # --------------------------------------------------------
    # RAW PRODUCT FRAME
    # --------------------------------------------------------

    raw_product = pd.DataFrame(
        [
            {
                "title":
                    title,

                "category_name":
                    category_name,

                "stars":
                    stars,

                "reviews":
                    reviews,

                "boughtInLastMonth":
                    bought_in_last_month,

                "isBestSeller":
                    is_best_seller,

                "cluster_id":
                    cluster_id,
            }
        ]
    )


    # --------------------------------------------------------
    # FEATURE ENGINEERING
    # --------------------------------------------------------

    product_df = (
        create_price_features(
            raw_product
        )
    )


    # --------------------------------------------------------
    # STRUCTURED FEATURES
    # --------------------------------------------------------

    structured_features = (
        price_preprocessor
        .transform(
            product_df
        )
    )


    # --------------------------------------------------------
    # TF-IDF TITLE FEATURES
    # --------------------------------------------------------

    text_features = (
        price_tfidf
        .transform(
            product_df[
                "title"
            ]
        )
    )


    # --------------------------------------------------------
    # FINAL FEATURE VECTOR
    # --------------------------------------------------------

    final_features = hstack(
        [
            csr_matrix(
                structured_features
            ),

            text_features,
        ],

        format="csr",
    )


    # --------------------------------------------------------
    # LIGHTGBM PREDICTION
    # --------------------------------------------------------

    predicted_log_price = float(
        price_model
        .predict(
            final_features
        )[0]
    )


    # --------------------------------------------------------
    # REVERSE LOG1P
    # --------------------------------------------------------

    predicted_price = float(
        np.expm1(
            predicted_log_price
        )
    )


    predicted_price = max(
        0.0,
        predicted_price,
    )


    # --------------------------------------------------------
    # RESPONSE
    # --------------------------------------------------------

    return {

        "predicted_price":
            round(
                predicted_price,
                2,
            ),

        "cluster_id":
            cluster_id,

        "model_version":
            "final_clean_v1",

        "device":
            DEVICE,
    }


# ============================================================
# READY
# ============================================================

print()
print("=" * 80)
print("PRODUCTION INFERENCE PIPELINE READY")
print("=" * 80)

print(
    "✅ Clean cluster generated automatically"
)

print(
    "✅ No price used for clustering"
)

print(
    "✅ No price_band used"
)

print(
    "✅ No listPrice used"
)

print(
    "✅ Final clean LightGBM loaded"
)

PROJECT CONFIGURATION
Current directory: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/notebooks
Project root: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System

MODEL ARTIFACT CHECK
✅ Price model: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/final_clean_v1/price_model.joblib
✅ Price preprocessor: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/final_clean_v1/structured_preprocessor.joblib
✅ TF-IDF: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/final_clean_v1/title_tfidf.joblib
✅ Clean cluster scaler: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/clustering_clean/structured_scaler_clean.joblib
✅ Clean KMeans: /Us

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


✅ Production inference models loaded

MODEL DIMENSION CHECK
PCA input: 384
PCA output: 64
Cluster scaler input: 4
KMeans input: 68
Clusters: 100
✅ Model dimensions validated

PRODUCTION INFERENCE PIPELINE READY
✅ Clean cluster generated automatically
✅ No price used for clustering
✅ No price_band used
✅ No listPrice used
✅ Final clean LightGBM loaded


In [10]:
result = predict_product_price(

    title=(
        "Dell Latitude Laptop "
        "Intel Core i7 16GB RAM "
        "512GB SSD 15.6 Inch"
    ),

    category_name=(
        "Computers & Tablets"
    ),

    stars=4.5,

    reviews=850,

    bought_in_last_month=100,

    is_best_seller=False,
)

print(result)

{'predicted_price': 467.51, 'cluster_id': 62, 'model_version': 'final_clean_v1', 'device': 'mps'}
